## Step A — Reshape Raw Trade Data to a Monthly Long Panel

### Objective
The goal of this step is to convert the raw USITC China import data from a **wide, year-level format** into a **monthly long-format panel**, where each row corresponds to a unique `(HTS10 product, month)` pair.

This step is purely mechanical: no aggregation, no treatment assignment, and no tariff data are used yet. The output of this step serves as the foundation for all subsequent cleaning and analysis.

---

### Raw Input Structure
The raw trade file (`China_Imports_1996_2005.csv`) is structured at the **product–year** level, with monthly quantities stored in separate columns:

- `hts10`: 10-digit Harmonized Tariff Schedule code  
- `year`: calendar year  
- `jan`–`dec`: first-unit quantities reported for each month  
- `quantity_description`: unit of measurement for quantities  
- `customs_value`: import value (yearly or monthly, depending on file version)

Key issues to address:
- HTS codes must be preserved as **strings** to avoid losing leading zeros.
- Month information is implicit and must be reshaped into an explicit time variable.

---

### Transformation Performed
This step performs the following operations:

1. **Enforce data types**
   - Read `hts10` as a character string.
   - Ensure `year` is numeric.
   - Keep monthly quantity columns (`jan`–`dec`) as numeric.

2. **Reshape from wide to long**
   - Convert the 12 monthly quantity columns into two variables:
     - `month` (integer from 1 to 12)
     - `first_unit_qty` (reported quantity for that month)

3. **Construct a monthly date**
   - Create a monthly date variable in the format `YYYY-MM-01`.
   - The day component is fixed to `01` to represent a monthly panel.

---

### Output Structure
After this step, the dataset should have the following structure:

- `hts10` (character, length 10)
- `year` (integer)
- `month` (integer, 1–12)
- `date` (Date, YYYY-MM-01)
- `first_unit_qty` (numeric)
- `customs_value` (numeric)
- `quantity_description` (character)

Each row represents:
> Imports of a specific HTS10 product in a specific month, with reported quantity and value in the original unit.

---

### Sanity Checks
Before proceeding to unit consistency checks (Step B), verify that:

- There are no duplicate `(hts10, date)` pairs.
- The date range spans **1996-01 to 2005-12**.
- All HTS10 codes are exactly 10 characters long.
- The number of rows is approximately 12× the number of product–year observations (accounting for missing months).

Passing these checks confirms that the monthly panel has been constructed correctly.


In [1]:
import pandas as pd
import numpy as np

"""
Step A already completed by Vincent.
Only need to import panel_hts10_monthly.csv
"""

df = pd.read_csv("../transformed_data/panel_hts10_monthly.csv")

## Step B — Unit Consistency and Quantity Quality Control

### Objective
The objective of this step is to ensure that quantity measures are **unit-consistent over time** before any aggregation is performed.

While import values (`customs_value`) can always be safely summed, quantity measures become meaningless if a product switches reporting units (e.g., kilograms → grams, dozens → pairs). To prevent unit-mixing, we exclude HTS10 product codes that exhibit **documented unit switches**.

Unlike a de novo detection procedure, this step relies on a **pre-computed unit-switching diagnostic file** provided by the project lead, ensuring consistency across team members.

---

### Input Data
This step uses two inputs:

1. **Monthly HTS10-level trade panel**  
   (output of Step A)
   - `hts10`
   - `date`
   - `customs_value`
   - `first_unit_qty`
   - `unit_value`

2. **HTS10 unit-switch diagnostic file**  
   (leader-provided)
   - `hts10`
   - `from_quantity_description`
   - `to_quantity_description`
   - `switch_date`
   - Pre- and post-switch quantity and value diagnostics

The diagnostic file identifies HTS10 codes that change quantity units at a specific date and documents the magnitude of the resulting discontinuities.

---

### Interpretation of the Unit-Switch File
Each row in the diagnostic file corresponds to an HTS10 product code that **changes quantity units over time**.

Key fields include:
- `from_quantity_description` → original reporting unit  
- `to_quantity_description` → new reporting unit  
- `switch_date` → first month of the new unit  
- Pre- vs. post-switch medians for:
  - quantities
  - customs values
  - unit values

These diagnostics confirm that unit switches are often associated with large, mechanical breaks in measured quantities or unit values, making aggregation invalid.

---

### Exclusion Rule
The unit consistency rule enforced in this step is:

> **All HTS10 product codes appearing in the unit-switch diagnostic file are excluded from quantity-based analysis.**

Operationally:
- Any HTS10 code listed in the diagnostic file is dropped **entirely**, for all months.
- This exclusion is applied **prior to aggregation** to HS8 or higher levels.

This conservative rule ensures that:
- No quantity measure is ever constructed from mixed units.
- Later HS8-level quantities are internally coherent.

Import values remain valid and are unaffected by unit switching.

---

### Treatment of Zero-Trade Months
Months with zero reported trade (`first_unit_qty = 0` and `customs_value = 0`) are retained.

- Zero observations reflect legitimate “no import” months.
- They do not indicate unit switching.
- Retaining them preserves a balanced monthly panel for Difference-in-Differences estimation.

---

### Output of This Step
The output is a filtered monthly HTS10-level dataset in which:

- All remaining HTS10 codes have a stable quantity unit over time.
- Quantity measures are safe to aggregate within and across products.
- The time structure and value data remain unchanged.

This cleaned dataset serves as the direct input for **Step C (Aggregation to HS8 × Month)**.

---

### Diagnostics and Transparency
For reproducibility, we report:

- The number of HTS10 codes excluded due to unit switching.
- The share of total trade value affected by these exclusions.
- Confirmation that no remaining HTS10 code appears in the unit-switch list.

These diagnostics document the scope of the exclusion and ensure transparency of the cleaning procedure.


In [2]:
# Import diagnostic file
quant_switch = pd.read_csv("../transformed_data/quantity_description_switch_diagnostics.csv")

# Ensure hts10 is string
df["hts10"] = df["hts10"].astype(str)
quant_switch["hts10"] = quant_switch["hts10"].astype(str)

# Filter
df_clean = df[~df["hts10"].isin(quant_switch["hts10"].unique())]

## Step C — Aggregate Trade Data to HS8 × Month

### Objective
The objective of this step is to aggregate the cleaned monthly trade data from the **HTS10 × month** level to the **HS8 × month** level, which is the product–time unit used in the empirical analysis.

Aggregation is performed **after unit consistency filtering** (Step B) to ensure that all quantity measures are meaningful and comparable within each aggregated product group.

---

### Data Context
The input to this step is the filtered monthly HTS10-level dataset, in which all HTS10 codes exhibiting quantity unit switches have been removed.

Relevant variables include:
- `hts10`: 10-digit product code
- `date`: monthly date (`YYYY-MM-01`)
- `customs_value`: monthly import value
- `first_unit_qty`: monthly quantity in a stable unit
- `unit_value`: value per unit (when quantity > 0)

---

### Construction of HS8 Codes
HS8 product codes are constructed by truncating the HTS10 code:

`hs8 = first 8 digits of hts10`


All aggregation is performed at the `(hs8, date)` level.

---

### Aggregation Rules

Aggregation proceeds separately for values and quantities, following conservative and transparent rules:

#### Import Values
- `customs_value` is **summed** across all HTS10 codes belonging to the same HS8 product in a given month.
- Import values are always commensurable and safe to aggregate.

#### Quantities
- `first_unit_qty` is summed **only when quantities are defined and unit-consistent**.
- Because unit-switching HTS10 codes have already been removed, remaining quantities are internally consistent within each HS8–month cell.

If no positive quantity is observed for a given `(hs8, date)`, the aggregated quantity is recorded as zero.

#### Unit Values
- The HS8-level unit value is computed as: `unit_value = customs_value / first_unit_qty`


- Unit values are defined only when aggregated quantity is strictly positive; otherwise, they are left missing.

---

### Output Variables
The resulting HS8 × month dataset contains:

- `hs8`: 8-digit product code
- `date`: monthly date
- `customs_value`: total monthly import value
- `first_unit_qty`: total monthly quantity (when defined)
- `unit_value`: value per unit (when quantity > 0)

This dataset constitutes the **analysis-ready trade panel** prior to treatment assignment.

---

### Data Integrity Checks
To validate the aggregation, we verify that:

- There are no duplicate `(hs8, date)` observations.
- The monthly date range remains **1996-01 to 2005-12**.
- Aggregated quantities are non-negative.
- Unit values are only defined when quantities are positive.

These checks ensure that the aggregation step preserves both the time structure and the economic meaning of the data.

---

### Next Step
The HS8 × month panel constructed here serves as the input for **Step 2.4**, where pre-policy tariff exposure is merged and treatment indicators are constructed for Difference-in-Differences estimation.


In [3]:
# Create hs8 columns

df_clean["hs8"] = df_clean["hts10"].str[:8]

df_clean["hs8"]

/var/folders/hq/hfb1chlj2pxg9shnwqj9t7jw0000gn/T/ipykernel_97575/2084023629.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["hs8"] = df_clean["hts10"].str[:8]


0          10111002
1          10111002
2          10111002
3          10111002
4          10111002
             ...   
1251307    99999500
1251308    99999500
1251309    99999500
1251310    99999500
1251311    99999500
Name: hs8, Length: 1250592, dtype: object

In [4]:
# Check for quantity unit changes between hs8 before aggregation

df_raw = pd.read_csv("../raw_data/China_Imports_1996_2005.csv")

df_raw.rename(columns={"HTS Number": "hts10"}, inplace=True)

df_raw.columns

df_raw["hts10"] = df_raw["hts10"].astype(str)

# Exclude known violators based off of hts10
df_raw = df_raw[~df_raw["hts10"].isin(quant_switch["hts10"].unique())]

# Remove "Value for:" naming in Quantity Description
df_raw["Quantity Description"] = (
    df_raw["Quantity Description"]
    .str.replace("^Value for:\s*", "", regex=True)
    .str.lower()
)

# Extract just Quantity Description column
unit_map = (
    df_raw[["hts10", "Quantity Description"]]
    .dropna(subset=["Quantity Description"])
    .drop_duplicates()
)

# Confirm unique Quantity Description per hts10
assert (
    unit_map.groupby("hts10")["Quantity Description"].nunique().le(1).all()
), "Some HTS10 codes map to multiple quantity descriptions"

# Merge just the Quantity Description column into our clean dataframe based on hts10 code
df_clean["hts10"] = df_clean["hts10"].astype(str)
unit_map["hts10"] = unit_map["hts10"].astype(str)

if "Quantity Description" not in df_clean.columns:
    df_clean = df_clean.merge(
        unit_map,
        on="hts10",
        how="left",
        validate="many_to_one"
    )

df_clean.columns


/var/folders/hq/hfb1chlj2pxg9shnwqj9t7jw0000gn/T/ipykernel_97575/1936762793.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["hts10"] = df_clean["hts10"].astype(str)


Index(['Country', 'Year', 'Month', 'hts10', 'customs_value', 'first_unit_qty',
       'date', 'unit_value', 'ln_customs_value', 'ln_first_unit_qty',
       'ln_unit_value', 'hs8', 'Quantity Description'],
      dtype='object')

In [5]:
# Check hs8 unique quantity description

unit_map_hs8 = (
    df_clean[["hs8", "Quantity Description"]]
    .dropna(subset=["Quantity Description"])
    .drop_duplicates()
)

# Count number of unique quantity descriptions per HS8
hs8_conflicts = (
    unit_map_hs8
    .groupby("hs8")["Quantity Description"]
    .nunique()
    .reset_index(name="n_unique_units")
)

# Keep only problematic HS8 codes
hs8_conflicts = hs8_conflicts[hs8_conflicts["n_unique_units"] > 1]

hs8_conflicts

conflict_rows = unit_map_hs8[
    unit_map_hs8["hs8"].isin(hs8_conflicts["hs8"])
].sort_values(["hs8", "Quantity Description"])

conflict_rows


,hs8,Quantity Description
57300,14012020,no units collected
57264,14012020,number
94956,23099010,kilograms
95160,23099010,metric tons
129636,28444000,megabecquerels
...,...,...
1242300,96151960,no units collected
1244352,97050000,component grams
1244436,97050000,no units collected
1245024,98010010,kilograms


In [6]:
# Replace "no units collected" with NaN, create real_unit column
df_clean["real_unit"] = np.where(
    df_clean["Quantity Description"] == "no units collected",
    np.nan,
    df_clean["Quantity Description"]
)

# Count number of unique quantity descriptions. Remove rows with NaN units. Name new column "n_real_units"
unit_counts = (
    df_clean
    .dropna(subset=["real_unit"])
    .groupby(["hs8", "date"])["real_unit"]
    .nunique()
    .rename("n_real_units")
    .reset_index()
)

# Sum customs_value
agg_value = (
    df_clean
    .groupby(["hs8", "date"], as_index=False)
    .agg(customs_value=("customs_value", "sum"))
)

# Sum quantities only if row has valid units. Temporarily use first unit in list of units used
agg_qty = (
    df_clean
    .dropna(subset=["real_unit"])
    .groupby(["hs8", "date"], as_index=False)
    .agg(
        first_unit_qty=("first_unit_qty", "sum"),
        quantity_unit=("real_unit", "first")
    )
)

# Merge everything
df_agg = (
    agg_value
    .merge(unit_counts, on=["hs8", "date"], how="left")
    .merge(agg_qty, on=["hs8", "date"], how="left")
)

# Set HS8-months with no real units unit count to 0
df_agg["n_real_units"] = df_agg["n_real_units"].fillna(0)

# HS8-months with multiple units return False
df_agg["unit_consistent"] = df_agg["n_real_units"] <= 1

# Invalidate quantities in rows where unit_consistent == False
df_agg.loc[~df_agg["unit_consistent"], ["first_unit_qty", "quantity_unit"]] = np.nan

# Calcualte unit value where first_unit_quantity > 0
df_agg["unit_value"] = np.where(
    df_agg["first_unit_qty"] > 0,
    df_agg["customs_value"] / df_agg["first_unit_qty"],
    np.nan
)

# Calculate log normalized values
df_agg["ln_customs_value"] = np.log1p(df_agg["customs_value"])

df_agg["ln_first_unit_qty"] = np.where(
    df_agg["first_unit_qty"] >= 0,
    np.log1p(df_agg["first_unit_qty"]),
    np.nan
)

df_agg["ln_unit_value"] = np.where(
    df_agg["unit_value"] > 0,
    np.log(df_agg["unit_value"]),
    np.nan
)

In [7]:
df_agg.head(10)

,hs8,date,customs_value,n_real_units,first_unit_qty,quantity_unit,unit_consistent,unit_value,ln_customs_value,ln_first_unit_qty,ln_unit_value
0,10019020,2004-01-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
1,10019020,2004-02-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
2,10019020,2004-03-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
3,10019020,2004-04-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
4,10019020,2004-05-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
5,10019020,2004-06-01,2720,1.0,1600.0,kilograms,True,1.7,7.908755,7.378384,0.530628
6,10019020,2004-07-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
7,10019020,2004-08-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
8,10019020,2004-09-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN
9,10019020,2004-10-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN


## STEP D: Construct Pre-Policy NTR Gap Exposure (`gap_pre`)

### Objective
This step constructs a pre-determined treatment intensity measure at the HS8 level based on tariff policy prior to China’s WTO accession. We use the **1999 NTR gap** as a product-level exposure measure that is fixed over time.

---

### Data Source
Tariff information is taken from the AER 2013 replication files (`tar_val.dta`). The key variables used are:

- `hs8`: HS8 product code  
- `year`: calendar year  
- `ntr_rate`: NTR rate
- `nonntr_rate`: Non-NTR rate
- `spread`: NTR gap
- `v`: trade value weight

---

### Step D.1: Restrict to Pre-Policy Year
To ensure that treatment intensity is predetermined and not influenced by post-WTO adjustments, we restrict the tariff data to the year **1999**, the last full year prior to China’s WTO accession.

---

### Step D.2: Collapse Tariff Data to HS8
The tariff dataset may contain multiple observations per HS8 code. We collapse the data to one observation per HS8 by computing the average NTR gap. When trade values are available, we use a value-weighted average.

The resulting variable is:

- **`gap_pre`**: pre-policy NTR gap at the HS8 level

This variable is time-invariant and will be merged onto the monthly trade panel.

---

### Step D.3: Merge with HS8-Month Trade Panel
The pre-policy NTR gap is merged onto the HS8-month trade panel constructed in STEP C using the HS8 product code.

After the merge, each `(hs8, date)` observation is assigned the same pre-policy exposure measure.

---

### Output
The resulting dataset contains:
- HS8-month trade outcomes  
- Quantity consistency indicators  
- A time-invariant pre-policy NTR gap measure (`gap_pre`)  

This dataset is now ready for constructing treatment and post-policy indicators.

---

### Sanity Checks
Before proceeding, we verify that:
- `gap_pre` is constant within HS8 codes  
- The distribution of `gap_pre` is reasonable  
- No duplicate `(hs8, date)` observations are introduced during the merge


In [8]:
df_ntr = pd.read_stata('../transformed_data/data_files_aer_2013/tar_val.dta')
df_ntr.head(50)

,hs8num,hs8,ntr_rate,nonntr_rate,spread,v,year
0,1011100,01011100,0.000,0.00,0.000,4523119.0,1989.0
1,1011100,01011100,0.000,0.00,0.000,4768635.0,1990.0
2,1011100,01011100,0.000,0.00,0.000,1766068.0,1991.0
3,1011100,01011100,0.000,0.00,0.000,2912166.0,1992.0
4,1011100,01011100,0.000,0.00,0.000,3236751.0,1993.0
5,1011100,01011100,0.000,0.00,0.000,5486697.0,1994.0
6,1011100,01011100,0.000,0.00,0.000,9745475.0,1995.0
7,1011100,01011100,0.000,0.00,0.000,6436244.0,1996.0
8,1011100,01011100,0.000,0.00,0.000,11890543.0,1997.0
9,1011100,01011100,0.000,0.00,0.000,38334898.0,1998.0


In [9]:
# Exclude all post WTO accession (1999) rows

df_ntr = df_ntr[df_ntr["year"] < 2000]
df_ntr["year"].unique()

array([1989., 1990., 1991., 1992., 1993., 1994., 1995., 1996., 1997.,
       1998., 1999.], dtype=float32)

In [10]:
any(df_ntr["v"].isna())

False

In [11]:
# Exclude all rows where NTR-gap is NaN
df_ntr = df_ntr[~df_ntr["spread"].isna()]

In [12]:
# Remove any rows that have no trade value
df_ntr = df_ntr[df_ntr["v"] > 0]

In [13]:
# Calculate weighted average of NTR-gap per hs8

gap_pre = (
    df_ntr
    .groupby("hs8")
    .apply(lambda x: np.average(x["spread"], weights=x["v"]))
    .rename("gap_pre")
    .reset_index()
)

gap_pre

/var/folders/hq/hfb1chlj2pxg9shnwqj9t7jw0000gn/T/ipykernel_97575/3045484414.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: np.average(x["spread"], weights=x["v"]))


,hs8,gap_pre
0,01011100,0.000000
1,01011900,0.200000
2,01012010,0.000000
3,01012020,0.040097
4,01012030,0.000000
...,...,...
11730,98170048,0.000000
11731,98170070,0.000000
11732,98180001,0.500000
11733,98180003,0.500000


In [14]:
# Merge with our main dataframe

df_agg = df_agg.merge(gap_pre, on="hs8", how="left")

df_agg.head(50)


,hs8,date,customs_value,n_real_units,first_unit_qty,quantity_unit,unit_consistent,unit_value,ln_customs_value,ln_first_unit_qty,ln_unit_value,gap_pre
0,10019020,2004-01-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
1,10019020,2004-02-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
2,10019020,2004-03-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
3,10019020,2004-04-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
4,10019020,2004-05-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
5,10019020,2004-06-01,2720,1.0,1600.0,kilograms,True,1.700000,7.908755,7.378384,0.530628,0.066773
6,10019020,2004-07-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
7,10019020,2004-08-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
8,10019020,2004-09-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
9,10019020,2004-10-01,0,1.0,0.0,kilograms,True,NaN,0.000000,0.000000,NaN,0.066773
